# Library Dataset — Pattern-Library Cards

Goal: produce 18-30 curated pattern-library cards for the backend schema
(`id`, `name`, `category`, `implies`, `why_misread` — no `source` field), covering 6 fixed
categories: Softened request, Vague deadline, Unassigned owner, Unclear priority, Undefined
scope, Feedback framing.

This notebook covers Stage 0 (methodology) and Stage 1 (Softened Request candidate pool from
the Stanford Politeness corpus) only, then pauses for manual selection of 5 candidates before
Stage 2.

## Stage 0 — Methodology

**Category design.** `Vague deadline`, `Unclear priority`, and `Undefined scope` are instances
of what Berry (Daniel M. Berry, "Ambiguity in Natural Language Requirements Documents",
University of Waterloo) calls **vagueness**: a category has no clear boundary between what
counts and what doesn't — his own example is *fast*, which "has no clear boundary between fast
and not fast." A deadline like "soon," a priority like "high," or a scope like "handle the
project" are vague in exactly this sense: there's no discrete line separating what satisfies
them from what doesn't.

`Unassigned owner` is a different kind of defect: information that is simply **missing** (who is
responsible), not information that is present but fuzzy. This is not part of Berry's ambiguity
taxonomy specifically — his taxonomy tree (Vagueness / Generality / Linguistic Ambiguity /
Software Engineering Ambiguity) doesn't include an "incompleteness" branch. Missing-information
defects are instead a separate, standard requirements-quality dimension recognized broadly in
the RE literature (e.g. IEEE 830 lists "unambiguous" and "complete" as two distinct SRS quality
criteria, not one). So: `Vague deadline` / `Unclear priority` / `Undefined scope` = vagueness
(Berry's sense, present-but-fuzzy); `Unassigned owner` = incompleteness (a different, standard
RE defect type, present-but-missing) — two different failure modes, not the same one applied
twice.

**Softened request / Feedback framing.** These draw on two linguistic literatures instead:
- Danescu-Niculescu-Mizil et al. (2013) on politeness strategies — negative-politeness devices
  (hedges, indirect phrasing, modal softeners) are speaker strategies for minimizing the
  apparent imposition of a request. The `hedge_wordlist`, `please`, and `counterfactual_modal`
  features built in Stage 4 of `Data_Cleaning.ipynb` operationalize exactly this.
- Grice's Cooperative Principle / conversational implicature — a softened or indirectly-framed
  request routinely says less than it means, relying on the hearer to infer the actual ask. This
  is precisely the gap this pattern library is trying to make explicit: what a message *says*
  (`name`) vs. what it typically *implies* (`implies`).

**Process, not a classifier.** This dataset is **not** the output of a trained model
auto-labeling text into these 6 categories. It's a curated pipeline:
1. Real-corpus candidate screening (Stage 1, below) — only possible for `Softened request`,
   since the Stanford corpus has no deadline/owner/priority/scope annotations.
2. Manual selection from the candidate pool (you).
3. Hand-designed candidates grounded in the vagueness/incompleteness distinction above, for the
   5 categories the corpus can't cover (Stage 2).
4. LLM-assisted drafting of `implies`/`why_misread` copy (Stage 3), against a fixed style
   reference.
5. Manual review of every card before export (Stage 3-4).

Every card's provenance is traceable: real-corpus-derived cards keep their original `Id` and
`Normalized Score` as a paper trail; hand-designed cards are labeled as such. Nothing is auto-
classified or silently generated without a review step.

## Stage 1 — Softened Request candidate pool

`Softened request` is the only category the Stanford Politeness corpus can support — it has no
annotations for deadlines, ownership, priority, or scope. Pulling candidates from
`stanford_politeness_features.csv` (the feature-augmented export from `Data_Cleaning.ipynb`):

**Filter:** `hedge_wordlist` OR `please` OR `counterfactual_modal` is True (any one of the three
softening markers), AND `is_high_disagreement` is False AND `is_too_short` is False (exclude
low-quality/noisy samples per the flags built in Stage 2 of the cleaning notebook).

**Sort:** descending `Normalized Score`, so the most-polite-per-annotators requests surface
first — these are the strongest real-world examples of the softening strategies actually
working. Top 30 shown below for manual selection; nothing is auto-picked.

In [1]:
import json
import pandas as pd

pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 160)

df = pd.read_csv('stanford_politeness_features.csv')

softening_mask = df['hedge_wordlist'] | df['please'] | df['counterfactual_modal']
quality_mask = (~df['is_high_disagreement']) & (~df['is_too_short'])
candidates = df[softening_mask & quality_mask].copy()

print(f"total rows: {len(df)}")
print(f"pass softening filter (hedge_wordlist OR please OR counterfactual_modal): {softening_mask.sum()}")
print(f"pass quality filter (not high-disagreement, not too-short): {quality_mask.sum()}")
print(f"pass both: {len(candidates)}")

def matched_markers(row):
    hits = []
    if row['hedge_wordlist']: hits.append('hedge_wordlist')
    if row['please']: hits.append('please')
    if row['counterfactual_modal']: hits.append('counterfactual_modal')
    return ', '.join(hits)

candidates['matched_markers'] = candidates.apply(matched_markers, axis=1)
top30 = candidates.sort_values('Normalized Score', ascending=False).head(30)
top30[['Id', 'source', 'Normalized Score', 'matched_markers', 'Request']].reset_index(drop=True)

total rows: 10956
pass softening filter (hedge_wordlist OR please OR counterfactual_modal): 4191
pass quality filter (not high-disagreement, not too-short): 10463
pass both: 4015


,Id,source,Normalized Score,matched_markers,Request
0,7094551,stack-exchange,2.657228,hedge_wordlist,"Thank you, after some experimenting, this seems to be the way to go. Any thoughts on how to move the reparented component back to its original place in DOM (and ExtJS component hierarchy) once the..."
1,5961611,stack-exchange,2.643058,counterfactual_modal,Thank you very much for your comment! Could you help to mention an example of debugger helper?
2,18437,stack-exchange,2.558272,please,"@Kosmonaut: Yes please, I agree with Martha. What is the deal with W ?"
3,19538,wikipedia,2.468367,counterfactual_modal,"Hey, thanks for your contributions here :). Would you mind referencing them to conform with wiki-policy?"
4,568377,wikipedia,2.299406,"hedge_wordlist, please","Thank you much! May I call upon you if I make mistakes or need help, then, please?"
5,2272,stack-exchange,2.249884,"please, counterfactual_modal",Thank you very much for your answer. Could you please take a moment and provide the code I would need to add which would allow me to achieve these goals?
6,6702,stack-exchange,2.242102,hedge_wordlist,Sounds similar to this problem: http://cooking.stackexchange.com/questions/6433/why-does-my-velveeta-queso-dip-get-clumpier-the-longer-it-stays-in-the-microwave I guess Velvetta isn't designed to ...
7,2353305,stack-exchange,2.171490,"please, counterfactual_modal",@MitMaro Thanks. Could you please do the math of whether it's useful or not? :)
8,5854408,stack-exchange,2.169640,hedge_wordlist,Can you explain your scenario better? Perhaps with an example?
9,5146,stack-exchange,2.130009,hedge_wordlist,I think the question could bring some good answers. But maybe it's more fitting to make it a CW?


**Not auto-selecting.** Top 30 candidates above, sorted by politeness score — please pick 5
to carry into Stage 2. Pausing here.

## Stage 1 — selected

5 Softened request candidates confirmed:

| Id | source | Normalized Score | Request |
|---|---|---|---|
| 19538 | wikipedia | 2.468 | Hey, thanks for your contributions here :). Would you mind referencing them to conform with wiki-policy? |
| 2272 | stack-exchange | 2.250 | Thank you very much for your answer. Could you please take a moment and provide the code I would need to add which would allow me to achieve these goals? |
| 5854408 | stack-exchange | 2.170 | Can you explain your scenario better? Perhaps with an example? |
| 5816787 | stack-exchange | 1.959 | You're welcome. And now, could you give a more detailed description of the problem? |
| 7023 | stack-exchange | 2.033 | @dmckee: Thank you for replying. Can you please let me know the approach that should be adopted when building a software? |

## Stage 2 — candidates for the remaining 5 categories

The Stanford corpus has no deadline/owner/priority/scope annotations, so these are **hand-designed**,
not corpus-derived — grounded in the vagueness/incompleteness distinction from Stage 0:

- `Vague deadline`, `Unclear priority`, `Undefined scope` -> **vagueness** (Berry's sense: no clear
  boundary between what satisfies the requirement and what doesn't)
- `Unassigned owner` -> **incompleteness** (information — who's responsible — is simply missing)
- `Feedback framing` -> positive-politeness framing (Danescu-Niculescu-Mizil 2013): critique wrapped
  in gratitude/deference so the actual ask is softened, not omitted or fuzzy

5 candidates per category below — please pick 4 per category.

In [2]:
candidates_stage2 = [
    # Vague deadline (vagueness)
    ("Vague deadline", "vagueness", "Let's aim to have this done soon."),
    ("Vague deadline", "vagueness", "Can we wrap this up by early next week-ish?"),
    ("Vague deadline", "vagueness", "This should go out sometime before the launch."),
    ("Vague deadline", "vagueness", "Let's try to close this out in the next little while."),
    ("Vague deadline", "vagueness", "We'll need this fairly soon, so don't sit on it too long."),

    # Unassigned owner (incompleteness)
    ("Unassigned owner", "incompleteness", "This needs to be reviewed before we ship."),
    ("Unassigned owner", "incompleteness", "Someone should follow up with the client on this."),
    ("Unassigned owner", "incompleteness", "The onboarding doc needs an update."),
    ("Unassigned owner", "incompleteness", "This bug should get fixed before the next release."),
    ("Unassigned owner", "incompleteness", "We need to circle back on the vendor contract."),

    # Unclear priority (vagueness)
    ("Unclear priority", "vagueness", "This is fairly important, so let's not let it slip."),
    ("Unclear priority", "vagueness", "Not urgent, but worth getting to at some point."),
    ("Unclear priority", "vagueness", "This should probably be a priority, but use your judgment."),
    ("Unclear priority", "vagueness", "Let's try to prioritize this when we can."),
    ("Unclear priority", "vagueness", "This matters, just not sure how much compared to everything else."),

    # Undefined scope (vagueness)
    ("Undefined scope", "vagueness", "Can you clean up the onboarding flow a bit?"),
    ("Undefined scope", "vagueness", "Take a pass at improving the report."),
    ("Undefined scope", "vagueness", "See if you can tighten up the checkout process."),
    ("Undefined scope", "vagueness", "Let's polish this before it goes out."),
    ("Undefined scope", "vagueness", "Can you look into making the dashboard better?"),

    # Feedback framing (positive-politeness framing, not vagueness/incompleteness)
    ("Feedback framing", "positive-politeness framing", "This is great overall — just a couple of small things to tweak."),
    ("Feedback framing", "positive-politeness framing", "Really solid work here, though I did have one thought."),
    ("Feedback framing", "positive-politeness framing", "I love the direction — one thing I'm wondering about is the pacing."),
    ("Feedback framing", "positive-politeness framing", "Nice job on this! A few small suggestions below."),
    ("Feedback framing", "positive-politeness framing", "This is really close — just some minor polish needed."),
]

stage2_df = pd.DataFrame(candidates_stage2, columns=['category', 'ambiguity_type', 'candidate_text'])
stage2_df.index += 1
stage2_df

,category,ambiguity_type,candidate_text
1,Vague deadline,vagueness,Let's aim to have this done soon.
2,Vague deadline,vagueness,Can we wrap this up by early next week-ish?
3,Vague deadline,vagueness,This should go out sometime before the launch.
4,Vague deadline,vagueness,Let's try to close this out in the next little while.
5,Vague deadline,vagueness,"We'll need this fairly soon, so don't sit on it too long."
6,Unassigned owner,incompleteness,This needs to be reviewed before we ship.
7,Unassigned owner,incompleteness,Someone should follow up with the client on this.
8,Unassigned owner,incompleteness,The onboarding doc needs an update.
9,Unassigned owner,incompleteness,This bug should get fixed before the next release.
10,Unassigned owner,incompleteness,We need to circle back on the vendor contract.


**Not auto-selecting.** 25 hand-designed candidates above (5 per category) — please pick 4 per
category. Pausing here.

## Stage 3 — copywriting: tone calibration (3 cards)

**Note before drafting:** the "25 confirmed candidates" this stage is meant to build on include
20 hand-designed Stage 2 candidates (4 per category), but the specific 4-of-5 cuts per category
haven't been sent yet — only the Stage 1 5 are locked. Not blocking on this for the 3 calibration
cards below (picked from items very likely to survive any reasonable cut), but the remaining 22
cards can't start until that selection arrives.

**Style target:** low-pressure, guiding tone — like a friend translating subtext, not a system
flagging an issue. `implies` opens with a soft hedge ("This usually means...", "There's
likely...", "Often, this signals..."); `why_misread` opens by validating the literal reading
before pointing at where the real information lives ("It's easy to read this as...", "You might
take this at face value as..."). No "Read literally" or clinical framing. Neither field judges
sender or receiver — this is a structural communication pattern, not a mistake either side made.

3 cards below, one per category, to calibrate before drafting the remaining 22:
1. **Softened request** (Stage 1, real corpus — demonstrates the opener-trim rule)
2. **Unassigned owner** (incompleteness logic — different from vagueness)
3. **Feedback framing** (positive-politeness framing — different from both)

(`Vague deadline` isn't repeated here since your own worked example already covers it.)

In [3]:
calibration_cards = [
    {
        "category": "Softened request",
        "name": "Would you mind referencing them to conform with wiki-policy?",
        "name_note": "Trimmed from original Id 19538: \"Hey, thanks for your contributions here :). Would you mind referencing them to conform with wiki-policy?\" — dropped the unrelated opening thanks, kept the request sentence.",
        "implies": "This usually means they'd like it done \u2014 \u2018would you mind\u2019 is softening the ask, not really leaving it optional.",
        "why_misread": "You might take \u2018would you mind\u2019 at face value as a genuine choice. Usually the answer they\u2019re hoping for is already decided.",
        "status": "draft",
    },
    {
        "category": "Unassigned owner",
        "name": "This bug should get fixed before the next release.",
        "name_note": None,
        "implies": "Often, this signals you're the one expected to own it \u2014 the passive wording just leaves your name out.",
        "why_misread": "It's easy to read this as a general note to the team. Ownership is usually assumed here, not spelled out.",
        "status": "draft",
    },
    {
        "category": "Feedback framing",
        "name": "This is great overall \u2014 just a couple of small things to tweak.",
        "name_note": None,
        "implies": "There's likely more substance in the \u2018small things\u2019 than the phrase suggests \u2014 the praise is softening what follows.",
        "why_misread": "You might take \u2018small things\u2019 at face value as minor. The real feedback is often packed into that second half.",
        "status": "draft",
    },
]

for c in calibration_cards:
    print(f"=== {c['category']} ===")
    print("name:       ", c['name'], f"({len(c['name'])} chars)")
    if c['name_note']:
        print("name_note:  ", c['name_note'])
    print("implies:    ", c['implies'])
    print("why_misread:", c['why_misread'])
    print("status:     ", c['status'])
    print()

=== Softened request ===
name:        Would you mind referencing them to conform with wiki-policy? (60 chars)
name_note:   Trimmed from original Id 19538: "Hey, thanks for your contributions here :). Would you mind referencing them to conform with wiki-policy?" — dropped the unrelated opening thanks, kept the request sentence.
implies:     This usually means they'd like it done — ‘would you mind’ is softening the ask, not really leaving it optional.
why_misread: You might take ‘would you mind’ at face value as a genuine choice. Usually the answer they’re hoping for is already decided.
status:      draft

=== Unassigned owner ===
name:        This bug should get fixed before the next release. (50 chars)
implies:     Often, this signals you're the one expected to own it — the passive wording just leaves your name out.
why_misread: It's easy to read this as a general note to the team. Ownership is usually assumed here, not spelled out.
status:      draft

=== Feedback framing ===
name:   

**Pausing here for tone review** before drafting the remaining 22 cards, and still waiting on
the 4-per-category selection for Stage 2's 20 candidates.

## Stage 3 — all 25 cards drafted

Dash rule applied retroactively to the 3 calibration cards too (their first drafts had em-dashes
in `implies`/`why_misread`, split into two sentences instead). `name` fields are left untouched
even where the original text contains a dash — the rule only applies to `implies`/`why_misread`.

Stage 1 opener-trims (same treatment as Id 19538 from the calibration round):
- Id 5816787: dropped "You're welcome. And now," kept the request sentence.
- Id 5854408: no unrelated opener to trim, used as-is.
- Id 7023: dropped "@dmckee: Thank you for replying." kept the request sentence.
- Id 2272: dropped "Thank you very much for your answer." kept the request sentence.

Temp IDs: `A1`-`A5` = Stage 1 (real-corpus-derived), `B1`-`B20` = Stage 2 (hand-designed).

In [4]:
cards = [
    # ---- A1-A5: Stage 1, real-corpus-derived (Softened request) ----
    {
        "temp_id": "A1", "category": "Softened request",
        "name": "Would you mind referencing them to conform with wiki-policy?",
        "name_note": "Id 19538. Dropped opening \"Hey, thanks for your contributions here :).\", kept the request sentence.",
        "implies": "This usually means they'd like it done. \u2018Would you mind\u2019 is softening the ask, not really leaving it optional.",
        "why_misread": "You might take \u2018would you mind\u2019 at face value as a genuine choice. Usually the answer they\u2019re hoping for is already decided.",
        "source": "real-corpus-derived",
    },
    {
        "temp_id": "A2", "category": "Softened request",
        "name": "Could you give a more detailed description of the problem?",
        "name_note": "Id 5816787. Dropped opening \"You're welcome. And now,\", kept the request sentence.",
        "implies": "Often, this signals they're stuck without more detail. \u2018Could you\u2019 is just making the ask sound less demanding.",
        "why_misread": "It's easy to read this as a casual follow-up question. Usually it's blocking them from moving forward.",
        "source": "real-corpus-derived",
    },
    {
        "temp_id": "A3", "category": "Softened request",
        "name": "Can you explain your scenario better? Perhaps with an example?",
        "name_note": "Id 5854408. No unrelated opener to trim, used as-is.",
        "implies": "This usually means they don't have enough to go on yet. \u2018Perhaps\u2019 is softening a fairly direct ask for detail.",
        "why_misread": "You might take \u2018perhaps with an example\u2019 at face value as a nice-to-have. Usually it's the thing that would actually help most.",
        "source": "real-corpus-derived",
    },
    {
        "temp_id": "A4", "category": "Softened request",
        "name": "Can you please let me know the approach that should be adopted when building a software?",
        "name_note": "Id 7023. Dropped opening \"@dmckee: Thank you for replying.\", kept the request sentence.",
        "implies": "There's likely a real decision waiting on this. \u2018Please let me know\u2019 is softening what's actually a request for direction.",
        "why_misread": "You might take this at face value as a casual question you can answer whenever. Usually something is waiting on your answer.",
        "source": "real-corpus-derived",
    },
    {
        "temp_id": "A5", "category": "Softened request",
        "name": "Could you please take a moment and provide the code I would need to add which would allow me to achieve these goals?",
        "name_note": "Id 2272. Dropped opening \"Thank you very much for your answer.\", kept the request sentence.",
        "implies": "There's likely a fairly concrete ask underneath all the softening. They want the actual code, not just general pointers.",
        "why_misread": "It's easy to read all the polite phrasing as low-pressure or optional. The request itself is usually pretty specific.",
        "source": "real-corpus-derived",
    },

    # ---- B1-B4: Vague deadline (vagueness) ----
    {
        "temp_id": "B1", "category": "Vague deadline",
        "name": "Let's aim to have this done soon.",
        "name_note": None,
        "implies": "There's often a real timeframe in mind. \u2018Soon\u2019 is doing the job of softening it, not removing it.",
        "why_misread": "It's easy to take \u2018soon\u2019 as low-pressure or open-ended. The actual timing is usually somewhere else, in a calendar or someone's head.",
        "source": "hand-designed",
    },
    {
        "temp_id": "B2", "category": "Vague deadline",
        "name": "Can we wrap this up by early next week-ish?",
        "name_note": None,
        "implies": "This usually means there's a specific day in mind, even if it isn't stated. \u2018-ish\u2019 is softening the ask, not the actual target.",
        "why_misread": "You might take \u2018early next week-ish\u2019 at face value as flexible. Usually there's a firmer date behind it than the wording suggests.",
        "source": "hand-designed",
    },
    {
        "temp_id": "B3", "category": "Vague deadline",
        "name": "This should go out sometime before the launch.",
        "name_note": None,
        "implies": "Often, this signals there's a real internal deadline tied to launch prep. \u2018Sometime before\u2019 just isn't naming it directly.",
        "why_misread": "It's easy to read this as loosely timed, whenever it fits. Usually it needs to land earlier than that to leave room for launch prep.",
        "source": "hand-designed",
    },
    {
        "temp_id": "B4", "category": "Vague deadline",
        "name": "We'll need this fairly soon, so don't sit on it too long.",
        "name_note": None,
        "implies": "This usually means the timeline is tighter than \u2018fairly soon\u2019 sounds. The softness is about tone, not the actual urgency.",
        "why_misread": "You might take this at face value as a gentle, no-rush reminder. Usually it's closer to a real deadline than it reads.",
        "source": "hand-designed",
    },

    # ---- B5-B8: Unassigned owner (incompleteness) ----
    {
        "temp_id": "B5", "category": "Unassigned owner",
        "name": "This needs to be reviewed before we ship.",
        "name_note": None,
        "implies": "Often, this signals the review is expected to land on you. The sentence just doesn't say so directly.",
        "why_misread": "It's easy to read this as a general status update. Usually someone specific is meant to pick it up, and it's often the reader.",
        "source": "hand-designed",
    },
    {
        "temp_id": "B6", "category": "Unassigned owner",
        "name": "The onboarding doc needs an update.",
        "name_note": None,
        "implies": "This usually means someone specific is expected to make the update, most often whoever's reading it. The sentence just leaves that part out.",
        "why_misread": "You might take this at face value as background information, not a task. Usually it's a task, just missing a name.",
        "source": "hand-designed",
    },
    {
        "temp_id": "B7", "category": "Unassigned owner",
        "name": "This bug should get fixed before the next release.",
        "name_note": None,
        "implies": "Often, this signals you're the one expected to own it. The passive wording just leaves your name out.",
        "why_misread": "It's easy to read this as a general note to the team. Ownership is usually assumed here, not spelled out.",
        "source": "hand-designed",
    },
    {
        "temp_id": "B8", "category": "Unassigned owner",
        "name": "We need to circle back on the vendor contract.",
        "name_note": None,
        "implies": "Often, this signals someone needs to own following up, and that's frequently whoever gets the message. \u2018We\u2019 doesn't always mean everyone equally.",
        "why_misread": "It's easy to read \u2018we\u2019 as shared ownership across the group. Usually one person ends up carrying it, and it may be you.",
        "source": "hand-designed",
    },

    # ---- B9-B12: Unclear priority (vagueness) ----
    {
        "temp_id": "B9", "category": "Unclear priority",
        "name": "This is fairly important, so let's not let it slip.",
        "name_note": None,
        "implies": "This usually means it should rank above routine tasks, though exactly how high is left open. \u2018Fairly important\u2019 is doing a lot of work here.",
        "why_misread": "You might take \u2018fairly important\u2019 at face value as one item among many. It's often meant to stand out more than that.",
        "source": "hand-designed",
    },
    {
        "temp_id": "B10", "category": "Unclear priority",
        "name": "Not urgent, but worth getting to at some point.",
        "name_note": None,
        "implies": "There's likely a real point where this becomes urgent, even though the message doesn't say when. \u2018At some point\u2019 is a placeholder, not a shrug.",
        "why_misread": "It's easy to read this as low priority indefinitely. Usually there's a point where it matters more than it sounds right now.",
        "source": "hand-designed",
    },
    {
        "temp_id": "B11", "category": "Unclear priority",
        "name": "Let's try to prioritize this when we can.",
        "name_note": None,
        "implies": "Often, this signals it should move up the list soon, not whenever there's spare time. \u2018When we can\u2019 usually means sooner than it sounds.",
        "why_misread": "You might take \u2018when we can\u2019 at face value as open-ended. Usually there's more urgency behind it than the wording lets on.",
        "source": "hand-designed",
    },
    {
        "temp_id": "B12", "category": "Unclear priority",
        "name": "This matters, just not sure how much compared to everything else.",
        "name_note": None,
        "implies": "This usually means the person hasn't ranked it themselves either, but still wants it kept in view. It's not a low-priority signal.",
        "why_misread": "It's easy to read the uncertainty as permission to deprioritize it. Usually it just means the ranking hasn't been worked out yet.",
        "source": "hand-designed",
    },

    # ---- B13-B16: Undefined scope (vagueness) ----
    {
        "temp_id": "B13", "category": "Undefined scope",
        "name": "Can you clean up the onboarding flow a bit?",
        "name_note": None,
        "implies": "There's likely a specific set of issues in mind, even though \u2018a bit\u2019 doesn't say what they are. The scope is narrower than it sounds.",
        "why_misread": "You might take \u2018a bit\u2019 at face value as light, open-ended polish. Usually there's a more specific fix already in mind.",
        "source": "hand-designed",
    },
    {
        "temp_id": "B14", "category": "Undefined scope",
        "name": "Take a pass at improving the report.",
        "name_note": None,
        "implies": "Often, this signals a particular kind of improvement, not a full redesign. \u2018A pass\u2019 usually means a bounded round of changes.",
        "why_misread": "It's easy to read \u2018improving\u2019 as broad, do-whatever-you-think scope. Usually the person has a narrower change in mind.",
        "source": "hand-designed",
    },
    {
        "temp_id": "B15", "category": "Undefined scope",
        "name": "See if you can tighten up the checkout process.",
        "name_note": None,
        "implies": "This usually means a specific friction point, not the whole process end to end. \u2018Tighten up\u2019 is pointing at something narrower.",
        "why_misread": "You might take this at face value as a full review of checkout. Usually it's about one part that's already bothering someone.",
        "source": "hand-designed",
    },
    {
        "temp_id": "B16", "category": "Undefined scope",
        "name": "Can you look into making the dashboard better?",
        "name_note": None,
        "implies": "There's likely a particular thing that's not working well, even though \u2018better\u2019 sounds open-ended. The ask is usually more specific than it reads.",
        "why_misread": "It's easy to read \u2018better\u2019 as a broad, open invitation to redesign. Usually there's a narrower issue driving the request.",
        "source": "hand-designed",
    },

    # ---- B17-B20: Feedback framing (positive-politeness framing) ----
    {
        "temp_id": "B17", "category": "Feedback framing",
        "name": "This is great overall \u2014 just a couple of small things to tweak.",
        "name_note": None,
        "implies": "There's likely more substance in the \u2018small things\u2019 than the phrase suggests. The praise is softening what follows.",
        "why_misread": "You might take \u2018small things\u2019 at face value as minor. The real feedback is often packed into that second half.",
        "source": "hand-designed",
    },
    {
        "temp_id": "B18", "category": "Feedback framing",
        "name": "I love the direction \u2014 one thing I'm wondering about is the pacing.",
        "name_note": None,
        "implies": "Often, this signals the pacing is the main thing they'd like changed. The praise is genuine, but it's also softening the actual note.",
        "why_misread": "You might take \u2018one thing\u2019 at face value as a minor aside. Usually it's the part of the feedback that matters most.",
        "source": "hand-designed",
    },
    {
        "temp_id": "B19", "category": "Feedback framing",
        "name": "Nice job on this! A few small suggestions below.",
        "name_note": None,
        "implies": "This usually means the suggestions are worth taking seriously, not just optional extras. \u2018Small\u2019 is softening the framing, not the content.",
        "why_misread": "It's easy to read \u2018small suggestions\u2019 as things you can skip. Usually they're the actual feedback, just gently introduced.",
        "source": "hand-designed",
    },
    {
        "temp_id": "B20", "category": "Feedback framing",
        "name": "This is really close \u2014 just some minor polish needed.",
        "name_note": None,
        "implies": "There's likely a specific list of changes behind \u2018minor polish,\u2019 even if it isn't spelled out yet. \u2018Close\u2019 is encouragement, not a final verdict.",
        "why_misread": "You might take \u2018minor polish\u2019 at face value as nearly done. Usually there's still a real list of changes to work through.",
        "source": "hand-designed",
    },
]

for c in cards:
    c["status"] = "draft"

print(f"total cards: {len(cards)}")
print(f"by category:")
import collections
print(collections.Counter(c['category'] for c in cards))

total cards: 25
by category:
Counter({'Softened request': 5, 'Vague deadline': 4, 'Unassigned owner': 4, 'Unclear priority': 4, 'Undefined scope': 4, 'Feedback framing': 4})


### Automated dash check + word-count check

Checking `implies`/`why_misread` specifically (not `name`, which is allowed to keep dashes from
the original text) for any em-dash (\u2014) or double-hyphen, plus a word-count sanity check.

In [5]:
dash_chars = ['\u2014', '--']
violations = []
for c in cards:
    for field in ['implies', 'why_misread']:
        text = c[field]
        for d in dash_chars:
            if d in text:
                violations.append((c['temp_id'], field, d))

print(f"dash violations in implies/why_misread: {len(violations)}")
for v in violations:
    print(' -', v)

print()
print("word counts (flagging any implies/why_misread over 30 words):")
long_fields = []
for c in cards:
    for field in ['implies', 'why_misread']:
        n_words = len(c[field].split())
        if n_words > 30:
            long_fields.append((c['temp_id'], field, n_words))
print(f"over-30-word fields: {len(long_fields)}")
for f in long_fields:
    print(' -', f)

print()
print("name field length check (must be <= 120 chars for db.String(120)):")
over_120 = [(c['temp_id'], len(c['name'])) for c in cards if len(c['name']) > 120]
print(f"over-120-char names: {len(over_120)}")
for o in over_120:
    print(' -', o)

dash violations in implies/why_misread: 0

word counts (flagging any implies/why_misread over 30 words):
over-30-word fields: 0

name field length check (must be <= 120 chars for db.String(120)):
over-120-char names: 0


In [6]:
for c in cards:
    print(f"[{c['temp_id']}] {c['category']}  ({c['source']})")
    print("  name:       ", c['name'])
    if c['name_note']:
        print("  name_note:  ", c['name_note'])
    print("  implies:    ", c['implies'])
    print("  why_misread:", c['why_misread'])
    print("  status:     ", c['status'])
    print()

[A1] Softened request  (real-corpus-derived)
  name:        Would you mind referencing them to conform with wiki-policy?
  name_note:   Id 19538. Dropped opening "Hey, thanks for your contributions here :).", kept the request sentence.
  implies:     This usually means they'd like it done. ‘Would you mind’ is softening the ask, not really leaving it optional.
  why_misread: You might take ‘would you mind’ at face value as a genuine choice. Usually the answer they’re hoping for is already decided.
  status:      draft

[A2] Softened request  (real-corpus-derived)
  name:        Could you give a more detailed description of the problem?
  name_note:   Id 5816787. Dropped opening "You're welcome. And now,", kept the request sentence.
  implies:     Often, this signals they're stuck without more detail. ‘Could you’ is just making the ask sound less demanding.
  why_misread: It's easy to read this as a casual follow-up question. Usually it's blocking them from moving forward.
  status:     

**Not exporting yet.** All 25 cards above (`A1`-`A5` real-corpus, `B1`-`B20` hand-designed),
status `draft` on every one. Pausing for line-by-line review, especially the automated dash-check
result above.

## Stage 4 — Export

Renumbering `A1`-`A5`/`B1`-`B20` to final integer `id`s 1-25, grouped by category in the order
given: Softened request (1-5), Vague deadline (6-9), Unassigned owner (10-13), Unclear priority
(14-17), Undefined scope (18-21), Feedback framing (22-25). The `cards` list built in Stage 3 is
already in exactly this order, so the new id is just its 1-indexed position.

Two files:
- `library_dataset.json` — array of `{id, name, category, implies, why_misread}` only, matching
  the backend schema exactly (no `source`, no temp id, no `name_note`, no `status`).
- `library_dataset_preview.md` — the same 25 cards as a markdown table, keeping the source
  annotation (`real-corpus-derived` / `hand-designed` + original temp id) for team review /
  traceability, since that context is useful for humans even though it's not part of the schema.

In [7]:
VALID_CATEGORIES = [
    "Softened request", "Vague deadline", "Unassigned owner",
    "Unclear priority", "Undefined scope", "Feedback framing",
]

final_cards = []
for i, c in enumerate(cards, start=1):
    final_cards.append({
        "id": i,
        "name": c["name"],
        "category": c["category"],
        "implies": c["implies"],
        "why_misread": c["why_misread"],
        # kept only for the markdown preview, not written to the JSON schema
        "_temp_id": c["temp_id"],
        "_source": c["source"],
    })

# sanity check on the renumbering before export
expected_ranges = {
    "Softened request": range(1, 6), "Vague deadline": range(6, 10),
    "Unassigned owner": range(10, 14), "Unclear priority": range(14, 18),
    "Undefined scope": range(18, 22), "Feedback framing": range(22, 26),
}
for fc in final_cards:
    assert fc["id"] in expected_ranges[fc["category"]], f"id {fc['id']} out of expected range for {fc['category']}"
print("renumbering matches the requested category ranges: OK")

renumbering matches the requested category ranges: OK


In [8]:
json_records = [
    {k: c[k] for k in ("id", "name", "category", "implies", "why_misread")}
    for c in final_cards
]

with open("library_dataset.json", "w") as f:
    json.dump(json_records, f, indent=2, ensure_ascii=False)

print(f"wrote library_dataset.json with {len(json_records)} records")

wrote library_dataset.json with 25 records


In [9]:
md_lines = [
    "# Pattern Library — 25 Cards (Preview)",
    "",
    "For team review. `source` and original temp id kept here for traceability; not part of the",
    "exported JSON schema.",
    "",
    "| id | category | source | name | implies | why_misread |",
    "|---|---|---|---|---|---|",
]
for c in final_cards:
    source_label = f"{c['_source']} ({c['_temp_id']})"
    row = f"| {c['id']} | {c['category']} | {source_label} | {c['name']} | {c['implies']} | {c['why_misread']} |"
    md_lines.append(row)

with open("library_dataset_preview.md", "w") as f:
    f.write("\n".join(md_lines) + "\n")

print(f"wrote library_dataset_preview.md with {len(final_cards)} rows")

wrote library_dataset_preview.md with 25 rows


### Re-read + validate `library_dataset.json`

In [10]:
with open("library_dataset.json") as f:
    reloaded = json.load(f)

print(f"record count: {len(reloaded)}  (expected 25)")
assert len(reloaded) == 25

expected_fields = {"id", "name", "category", "implies", "why_misread"}
field_mismatches = [r["id"] for r in reloaded if set(r.keys()) != expected_fields]
print(f"records with unexpected field set: {len(field_mismatches)}")

empty_fields = []
for r in reloaded:
    for field in expected_fields:
        val = r[field]
        if val is None or (isinstance(val, str) and val.strip() == ""):
            empty_fields.append((r["id"], field))
print(f"empty/null field values: {len(empty_fields)}")
for e in empty_fields:
    print(' -', e)

bad_categories = [(r["id"], r["category"]) for r in reloaded if r["category"] not in VALID_CATEGORIES]
print(f"records with a category outside the 6 fixed values: {len(bad_categories)}")
for b in bad_categories:
    print(' -', b)

ids = [r["id"] for r in reloaded]
dup_ids = [i for i in set(ids) if ids.count(i) > 1]
print(f"duplicate ids: {len(dup_ids)}")

ids_sorted = sorted(ids)
print(f"ids are exactly 1..25 with no gaps: {ids_sorted == list(range(1, 26))}")

print()
print("=== ALL CHECKS ===")
all_ok = (
    len(reloaded) == 25
    and len(field_mismatches) == 0
    and len(empty_fields) == 0
    and len(bad_categories) == 0
    and len(dup_ids) == 0
    and ids_sorted == list(range(1, 26))
)
print("ALL PASS" if all_ok else "SOME CHECKS FAILED — see above")

record count: 25  (expected 25)
records with unexpected field set: 0
empty/null field values: 0
records with a category outside the 6 fixed values: 0
duplicate ids: 0
ids are exactly 1..25 with no gaps: True

=== ALL CHECKS ===
ALL PASS


### First 3 records of `library_dataset.json`

In [11]:
print(json.dumps(reloaded[:3], indent=2, ensure_ascii=False))

[
  {
    "id": 1,
    "name": "Would you mind referencing them to conform with wiki-policy?",
    "category": "Softened request",
    "implies": "This usually means they'd like it done. ‘Would you mind’ is softening the ask, not really leaving it optional.",
    "why_misread": "You might take ‘would you mind’ at face value as a genuine choice. Usually the answer they’re hoping for is already decided."
  },
  {
    "id": 2,
    "name": "Could you give a more detailed description of the problem?",
    "category": "Softened request",
    "implies": "Often, this signals they're stuck without more detail. ‘Could you’ is just making the ask sound less demanding.",
    "why_misread": "It's easy to read this as a casual follow-up question. Usually it's blocking them from moving forward."
  },
  {
    "id": 3,
    "name": "Can you explain your scenario better? Perhaps with an example?",
    "category": "Softened request",
    "implies": "This usually means they don't have enough to go on yet. 